# NB-04: Numerical Consistency Checker

Verifies that key numerical claims in the abstract, body text, and tables are internally consistent. Flags the 70 vs 71 task discrepancy, five-stage vs five-layer terminology, and validates timing claim arithmetic.

**Status as of 2026-07-03 audit round:**
- **FIX-N1, FIX-N2** — RESOLVED / confirmed false positive. No action needed.
- **FIX-N3** — split status: paper prose is settled (dual-threshold 33.3%/91.7% stated directly, §10.8), but the seed=123 rerun file (`exp_nguyen12_seed123_merged.json`) is still not committed. Step 6a below stays a stub until it exists.
- **FIX-C3 (3rd re-open, RESOLVED 2026-07-03)** — the Feynman split-protocol saga is closed out. Two prior 'corrected' values (106/180, then 71/90) were themselves stale duplicate-counting artifacts and are now dead ends. The paper's live, self-verification-gated figures (tab:provenance, strict R²≥0.999999) are **12/30 (40.0%) under random 80/20** and **13/30 (43.3%) under PCA 40/60**. The legacy 9/30 figure survives only as a labeled historical table entry (§10.7, `tab:feynman30-legacy`) and is not cited as a live claim anywhere else. TRUTH dict below is re-anchored to 12/30 and 13/30.
- **FIX-D1 (NEW, critical, OPEN)** — the DeFi §hybrid-attribution-bug disclosure. Paper still prints the pre-bug headline numbers (89.2%, +38.1pp hard-tier gain, 'zero catastrophic failures') inline, immediately followed by a footnote/bracketed correction (60.8%, −4.8pp, 22/74 masked failures). Both readings legitimately co-occur in the same sentence — the plain substring checks in Step 2 cannot tell 'number is correct' from 'number is flagged as superseded right next to it'. Step 6c below adds a footnote-aware check plus a corrected-figures verification against the DeFi benchmark output.
- **FIX-N4 (NEW, open, investigation stub)** — registry claims a stray '0.678' baseline co-occurs with '9/30' somewhere in §10.7. Step 6d below searches the source directly; as of this run, no occurrence of '0.678' was found anywhere in the .tex file, so the premise of this issue could not be reproduced against current source.

## Step 1 — Load ground-truth values

In [1]:
import re
import json
import os
from pathlib import Path

TEX_FILE = "jmlr_paper_main.tex"  # matches CI staging in ci_paper_notebooks.yml (FIX-XR4 corrected 2026-07-03: the 2026-06-28 owner note was wrong, contradicted by CI + paper_targets.json).
source = Path(TEX_FILE).read_text(encoding="utf-8")
lines  = source.splitlines()

# Ground-truth values from v3.0 benchmark (from tables in paper)
#
# FIX-C3 (RESOLVED, 3rd re-open, 2026-07-03): feynman_successes/feynman_total now
# re-anchored to the paper's live, self-verification-gated tab:provenance figures.
# Both prior 'fixes' stored here (106/180, then 71/90-equivalent 142/180) were
# themselves stale duplicate-counting artifacts and must NOT be reintroduced.
# Current truth: 12/30 (40.0%) random 80/20, 13/30 (43.3%) PCA 40/60.
#
# FIX-D1 (OPEN, critical): the DeFi hybrid-attribution-bug means every
# hybrid.success-derived headline number below is the PRE-BUG (uncorrected)
# reading. The paper discloses both readings side by side via inline
# footnotes/bracketed corrections (see §hybrid-attribution-bug). The
# *_corrected keys are the paper's disclosed corrected values; the plain
# keys are the uncorrected headline values that still appear in the abstract.
TRUTH = {
    "hypatix_success_pct":              89.2,   # UNCORRECTED headline (abstract) — see FIX-D1
    "hypatix_success_pct_corrected":     60.8,   # FIX-D1 disclosed correction (45/74)
    "llm_success_pct":                  62.2,
    "nn_success_pct":                    5.4,
    "hypatix_gain_pp":                  27.0,
    "hypatix_gain_over_nn_pp":          83.8,   # no corrected figure disclosed for this metric yet
    "hypatix_catastrophic":               0,     # UNCORRECTED headline ('zero catastrophic failures') — see FIX-D1
    "hypatix_catastrophic_masked_count": 22,     # FIX-D1: 22/74 tasks route through a sub-method that itself failed catastrophically (masked, not eliminated)
    "hypatix_hard_tier_gain_pp":         38.1,   # UNCORRECTED headline gain on hard tasks — see FIX-D1
    "hypatix_hard_tier_gain_pp_corrected": -4.8, # FIX-D1: corrected hard-tier delta is an actual LOSS vs LLM baseline
    "llm_catastrophic":                  6,
    "speedup_llm_routed":               1.73,
    "llm_routed_n":                      68,
    "feynman_successes":                 12,    # FIX-C3 RESOLVED (3rd re-open): random 80/20, tab:provenance, self-verification-gated
    "feynman_total":                     30,
    "feynman_pca4060_successes":         13,    # FIX-C3: PCA-directed 40/60 split, self-verification-gated
    "feynman_pca4060_total":             30,
    "nguyen_hyp":                        11,
    "nguyen_pysr":                       10,
    "nguyen_total":                      12,
    "easy_n":                            24,
    "medium_n":                          29,
    "hard_n":                            21,
    "total_tasks":                       74,
}

print("Ground-truth values loaded:")
for k, v in TRUTH.items():
    print(f"  {k:<38} = {v}")

# Resolve RESULTS_DIR for FIX-C3 / FIX-D1 checks further below
SCRIPT_DIR  = Path().resolve()
REPO_ROOT   = SCRIPT_DIR.parent if (SCRIPT_DIR.parent / "hypatiax").exists() else SCRIPT_DIR
RESULTS_DIR = Path(os.environ.get("RESULTS_DIR",
                   str(REPO_ROOT / "hypatiax/data/results")))
print(f"\nREPO_ROOT   : {REPO_ROOT}")
print(f"RESULTS_DIR : {RESULTS_DIR}")


Ground-truth values loaded:
  hypatix_success_pct                 = 89.2
  llm_success_pct                     = 62.2
  nn_success_pct                      = 5.4
  hypatix_gain_pp                     = 27.0
  hypatix_gain_over_nn_pp             = 83.8
  hypatix_catastrophic                = 0
  llm_catastrophic                    = 6
  speedup_llm_routed                  = 1.73
  llm_routed_n                        = 68
  feynman_successes                   = 9
  feynman_total                       = 30
  nguyen_hyp                          = 11
  nguyen_pysr                         = 10
  nguyen_total                        = 12
  easy_n                              = 24
  medium_n                            = 29
  hard_n                              = 21
  total_tasks                         = 74


## Step 2 — Abstract claim verification

In [ ]:
# Check abstract claims against ground truth
abstract_match = re.search(
    r'\\begin\{abstract\}(.*?)\\end\{abstract\}', source, re.DOTALL)
abstract = abstract_match.group(1) if abstract_match else ""

# UPDATED 2026-07-03 (FIX-C3 3rd re-open + FIX-D1): the legacy '9/30' figure is no
# longer a live abstract claim -- the abstract now reports the dual-protocol Feynman
# result directly as '12/30' (random 80/20) and '13/30' (PCA 40/60). Checks below
# reflect that. The '89.2' / '38.1' checks remain because those PRE-BUG headline
# numbers are still printed in the abstract by design, immediately followed by an
# inline footnote disclosing the FIX-D1 correction -- a plain substring check cannot
# tell 'this number is correct' from 'this number is flagged as superseded right next
# to it', so passing here is NOT confirmation the headline number is accurate. See
# Step 6c for a footnote-aware version of this same check.
checks = [
    ("89.2",  "89.2% near-perfect success rate in abstract (UNCORRECTED headline, see FIX-D1 / Step 6c)"),
    ("62.2",  "62.2% LLM baseline in abstract"),
    ("27",    "+27 pp gain in abstract"),
    ("83.8",  "+83.8 pp over NN in abstract"),
    ("1.73",  "1.73x speedup in abstract"),
    ("68",    "68/74 LLM-routed in abstract"),
    ("11/12", "Nguyen 11/12 in abstract"),
    ("12/30", "Feynman 12/30 (random 80/20) in abstract -- FIX-C3 re-anchored value"),
    ("13/30", "Feynman 13/30 (PCA 40/60) in abstract -- FIX-C3 re-anchored value"),
    ("38.1",  "+38.1 pp hard tasks in abstract (UNCORRECTED headline, see FIX-D1 / Step 6c)"),
]
print("Abstract claims verification:")
print("-" * 80)
for val, desc in checks:
    found = val in abstract
    status = "OK" if found else "MISSING"
    print(f"  [{status}]  {desc}")

legacy_930_in_abstract = "9/30" in abstract
print()
print(f"  [{'WARN' if legacy_930_in_abstract else 'OK'}]  Legacy '9/30' "
      f"{'still present' if legacy_930_in_abstract else 'absent'} in abstract "
      f"(expected absent -- FIX-C3 says legacy value should only survive as a\n"
      f"          historical table entry in §10.7, not in the abstract).")


## Step 3 — 70 vs 71 task discrepancy (instability section)

In [3]:
# Known issue: instability section says '70 tasks' in table caption
# but '71 cases' in body text
import re

instab_section = ""
in_sec = False
for ln in lines:
    if "Stability Under Stochastic" in ln:
        in_sec = True
    if in_sec:
        instab_section += ln + "\n"
    if in_sec and r"\subsection" in ln and "Stability" not in ln:
        break

hits_70 = [(i+1, ln.strip()) for i, ln in enumerate(lines)
           if ("70 tasks" in ln or "70 cases" in ln) and i > 0]
hits_71 = [(i+1, ln.strip()) for i, ln in enumerate(lines)
           if ("71 tasks" in ln or "71 cases" in ln) and i > 0]

print("Occurrences of '70 tasks/cases':")
for lno, ctx in hits_70:
    print(f"  line {lno}: {ctx[:100]}")
print()
print("Occurrences of '71 tasks/cases':")
for lno, ctx in hits_71:
    print(f"  line {lno}: {ctx[:100]}")

print()
if hits_70 and hits_71:
    print("CONFLICT: Both '70' and '71' appear.  Decide which is correct and unify.")
    print("  Table caption (tab:instability) says '70 tasks, K=30 runs each'")
    print("  Body text says 'correlation is computed across all 71 cases'")
    print("  -> The multi-run dataset has 70 tasks (4 tasks lack 30-run data);")
    print("     Spearman correlation footnote incorrectly says 71.")
    print("  ACTION: Change '71 cases' in body text to '70 tasks'.")

Occurrences of '70 tasks/cases':
  line 1608: Table~\ref{tab:instability} summarises the regime distribution across the 70 tasks
  line 1613: \caption{LLM instability regime distribution (70 tasks, $K=30$ runs each).

Occurrences of '71 tasks/cases':
  line 532: An earlier version of this benchmark comprised 71 tasks (Easy=24, Medium=27,
  line 1637: 71 cases, with A-Symbolic cases at ($\II=0$, $\Rsq=1$) and C-Collapse cases at

CONFLICT: Both '70' and '71' appear.  Decide which is correct and unify.
  Table caption (tab:instability) says '70 tasks, K=30 runs each'
  Body text says 'correlation is computed across all 71 cases'
  -> The multi-run dataset has 70 tasks (4 tasks lack 30-run data);
     Spearman correlation footnote incorrectly says 71.
  ACTION: Change '71 cases' in body text to '70 tasks'.


## Step 4 — Five-stage vs Five-layer terminology

In [4]:
# Known issue: Abstract says 'five-stage routing' but §8 says 'Five-Layer Architecture'
hits_stage = [(i+1, ln.strip()) for i, ln in enumerate(lines)
              if "five-stage" in ln.lower() or "five stage" in ln.lower()]
hits_layer = [(i+1, ln.strip()) for i, ln in enumerate(lines)
              if "five-layer" in ln.lower() or "five layer" in ln.lower()]

print("'five-stage' occurrences:")
for lno, ctx in hits_stage:
    print(f"  line {lno}: {ctx[:100]}")
print()
print("'five-layer' occurrences:")
for lno, ctx in hits_layer:
    print(f"  line {lno}: {ctx[:100]}")
print()
if hits_stage and hits_layer:
    print("INCONSISTENCY: 'five-stage routing' vs 'Five-Layer Architecture'.")
    print("  These describe the same system but use different terminology.")
    print("  ACTION: Standardise.  The abstract/intro use 'five-stage routing'.")
    print("  Section 8.3 heading 'Five-Layer Architecture Overview' should become")
    print("  'Five-Stage Routing Architecture' to match the abstract and §7.")

'five-stage' occurrences:
  line 143: addresses this through a five-stage routing and ensembling mechanism---trust
  line 232: \item \textbf{HypatiaX hybrid system}: a five-stage routing and ensembling
  line 698: \subsection{Component 3: Five-Stage Routing and Ensembling}
  line 854: \subsection{Five-Stage Architecture Overview}
  line 987: The full five-stage routing and ensembling system described in Section~\ref{sec:routing}.
  line 1786: \item \textbf{Cascade validation rather than single-criterion acceptance.} Our five-stage routing ca
  line 1795: HypatiaX builds on three lines of prior work. AI Feynman~\citep{udrescu2020feynman} uses neural netw

'five-layer' occurrences:
  line 872: Integrates all five layers.  Achieved near-zero extrapolation error on the

INCONSISTENCY: 'five-stage routing' vs 'Five-Layer Architecture'.
  These describe the same system but use different terminology.
  ACTION: Standardise.  The abstract/intro use 'five-stage routing'.
  Section 8.3 heading 'F

## Step 5 — Timing arithmetic cross-check

In [5]:
# Verify timing numbers internally consistent
timing_claims = {
    "mean_hybrid"  : (6.8,  "HypatiaX mean 6.8s"),
    "median_hybrid": (1.7,  "HypatiaX median 1.7s"),
    "mean_nn"      : (3.0,  "Neural MLP mean 3.0s"),
    "median_nn"    : (2.7,  "Neural MLP median 2.7s"),
    "speedup"      : (1.73, "1.73x speedup LLM-routed"),
    "mean_llm"     : (11.4, "Pure LLM mean 11.4s"),
}
print("Timing numbers check (each value must appear in paper):")
for key, (val, desc) in timing_claims.items():
    val_str = str(val)
    found = val_str in source
    status = "OK" if found else "MISSING"
    print(f"  [{status}]  {desc}")

print()
# Cross-check: 1.73x speedup from median NN 2.7s
estimated_speedup = 2.7 / 1.56  # median NN / estimated LLM-routed median
print(f"  Cross-check: NN median 2.7s / 1.73 = {2.7/1.73:.2f}s (should be ~1.56s for LLM-routed)")
print("  App C says 1.73x from 2.7s NN median -> LLM-routed median ~1.56s (marked with *)")

Timing numbers check (each value must appear in paper):
  [OK]  HypatiaX mean 6.8s
  [OK]  HypatiaX median 1.7s
  [OK]  Neural MLP mean 3.0s
  [OK]  Neural MLP median 2.7s
  [OK]  1.73x speedup LLM-routed
  [OK]  Pure LLM mean 11.4s

  Cross-check: NN median 2.7s / 1.73 = 1.56s (should be ~1.56s for LLM-routed)
  App C says 1.73x from 2.7s NN median -> LLM-routed median ~1.56s (marked with *)


## Step 6 — Fix recipe

## Step 6b — FIX-C3 (RESOLVED, 3rd re-open): Feynman dual-protocol result vs paper claim

Reads `exp2_random8020_summary.json` and `exp2_pca_4060_summary.json` (both written by `run_all.sh`) and compares the self-verification-gated solve rates against the re-anchored `TRUTH` values above (12/30 random 80/20, 13/30 PCA 40/60). Flags if either result file is missing, or if a result file's own numbers still match one of the now-dead-end stale values (106/180, 71/90-equivalent 142/180) rather than the current live figures.

In [ ]:
# FIX-C3 (RESOLVED, 3rd re-open): compare BOTH Feynman protocol results against TRUTH dict
RANDOM_8020_SUMMARY = (
    RESULTS_DIR
    / "comparison_results/feynman-tests/exp2_random8020/exp2_random8020_summary.json"
)
PCA_4060_SUMMARY = (
    RESULTS_DIR
    / "comparison_results/feynman-tests/exp2_pca_4060/exp2_pca_4060_summary.json"
)

# Dead-end values from prior re-opens of this issue -- must NOT reappear as a live result
STALE_VALUES = [(106, 180), (71, 90), (142, 180), (9, 30)]

print("FIX-C3: Feynman dual-protocol result check")
print("-" * 80)

protocol_checks = [
    ("random 80/20", RANDOM_8020_SUMMARY, "feynman_successes", "feynman_total"),
    ("PCA 40/60",     PCA_4060_SUMMARY,    "feynman_pca4060_successes", "feynman_pca4060_total"),
]

for label, path, truth_pass_key, truth_total_key in protocol_checks:
    print(f"\n  [{label}]")
    if not path.exists():
        print(f"    [SKIP] {path.name} not found -- experiment run not committed yet")
        continue

    data       = json.loads(path.read_text())
    n_pass     = data.get("n_pass")
    n_total    = data.get("n_total")
    rate       = data.get("solve_rate")
    protocol   = data.get("split_protocol", "?")
    truth_pass  = TRUTH[truth_pass_key]
    truth_total = TRUTH[truth_total_key]

    print(f"    File reports    : {n_pass}/{n_total} = {rate}  (protocol={protocol!r})")
    print(f"    TRUTH dict says : {truth_pass}/{truth_total} = {truth_pass/truth_total:.4f}")

    if (n_pass, n_total) in STALE_VALUES:
        print(f"    [FAIL] Result file reports a DEAD-END stale value {n_pass}/{n_total} -- "
              f"do not accept, re-derive from raw per-equation JSON per tab:provenance policy")
    elif n_pass == truth_pass and n_total == truth_total:
        print("    [OK]   Matches re-anchored TRUTH value")
    else:
        print("    [WARN] Result file does not match TRUTH -- investigate before citing in paper")

    result_str = f"{n_pass}/{n_total}"
    print(f"    Paper cites {result_str!r}: {result_str in source}")

print()
print("Reminder: legacy 9/30 and both prior 'corrected' values (106/180, 71/90) are dead ends.")
print("Only 12/30 (random 80/20) and 13/30 (PCA 40/60) are current, self-verification-gated figures.")


## Step 6c — FIX-D1 (OPEN, critical): DeFi hybrid-attribution-bug footnote-aware check

The abstract's substring checks in Step 2 can't distinguish a correct number from a pre-bug number sitting next to its own correction footnote. This step (1) verifies each disclosed correction actually appears near its uncorrected headline (footnote-aware), and (2) checks the corrected DeFi benchmark output file directly against the paper's disclosed corrected figures (60.8% overall, −4.8pp hard-tier delta, 22/74 masked catastrophic failures).

In [ ]:
# FIX-D1: footnote-aware check -- for each uncorrected headline number, verify a
# correction marker (a nearby 'flagged' / corrected value) is present in the same
# region of source, rather than just checking the headline number exists in isolation.
FOOTNOTE_PAIRS = [
    ("89.2",  "60.8",  "overall success rate"),
    ("38.1",  "4.8",   "hard-tier gain (pp)"),
]

print("FIX-D1: footnote-awareness check (uncorrected headline vs disclosed correction)")
print("-" * 80)
for uncorrected, corrected, label in FOOTNOTE_PAIRS:
    unc_idxs = [m.start() for m in re.finditer(re.escape(uncorrected), source)]
    found_paired = False
    for idx in unc_idxs:
        window = source[max(0, idx-50):idx+400]
        if corrected in window:
            found_paired = True
            break
    status = "OK" if found_paired else "WARN"
    print(f"  [{status}]  {label}: uncorrected {uncorrected!r} "
          f"{'is' if found_paired else 'is NOT'} paired with corrected {corrected!r} nearby")

catastrophic_masked_str = str(TRUTH["hypatix_catastrophic_masked_count"])
masked_disclosed = (catastrophic_masked_str in source) and ("masked" in source.lower())
print(f"  [{'OK' if masked_disclosed else 'WARN'}]  'zero catastrophic failures' headline is "
      f"paired with a '{catastrophic_masked_str}...masked' disclosure: {masked_disclosed}")

print()
print("FIX-D1: corrected DeFi benchmark file check")
print("-" * 80)
DEFI_CORRECTED_GLOB = list(RESULTS_DIR.glob("defi/hypatix_defi_benchmark_v3c_corrected_*.json"))
if not DEFI_CORRECTED_GLOB:
    print("  [MISSING] no hypatix_defi_benchmark_v3c_corrected_*.json found under RESULTS_DIR/defi/")
    print("  -> corrected figures (60.8% / -4.8pp / 22 masked) are asserted in the paper text")
    print("     but not yet backed by a committed, CI-checkable result file.")
else:
    defi_data = json.loads(DEFI_CORRECTED_GLOB[0].read_text())
    summary = defi_data.get("summary", {})
    checks = [
        ("corrected_success_rate",      TRUTH["hypatix_success_pct_corrected"] / 100, 0.005),
        ("hard_tier_gain_pp_corrected", TRUTH["hypatix_hard_tier_gain_pp_corrected"], 0.1),
        ("catastrophic_masked_count",  TRUTH["hypatix_catastrophic_masked_count"], 0),
    ]
    for key, expected, tol in checks:
        actual = summary.get(key)
        if actual is None:
            print(f"  [FAIL] summary.{key} missing from result file")
        elif abs(actual - expected) <= tol:
            print(f"  [OK]   summary.{key} = {actual} (expected {expected})")
        else:
            print(f"  [WARN] summary.{key} = {actual}, expected {expected} (tol {tol})")


## Step 6d — FIX-N4 (open, investigation stub): '9/30 vs 0.678' claim in §10.7

The registry flags a stray '0.678' baseline allegedly co-occurring with '9/30' somewhere in §10.7, but notes this has **not yet been investigated** and there is no detector regex for it. This cell does the first investigation step the registry asks for: search the actual source for '0.678' and report what (if anything) is found, rather than assuming the registry description is accurate.

In [ ]:
# FIX-N4: search for the alleged '0.678' figure anywhere in the source
hits_0678 = [(i+1, ln.strip()) for i, ln in enumerate(lines) if "0.678" in ln or "67.8" in ln]
hits_930  = [(i+1, ln.strip()) for i, ln in enumerate(lines) if "9/30" in ln]

print("FIX-N4 investigation:")
print("-" * 80)
print(f"  Occurrences of '0.678' / '67.8' in source : {len(hits_0678)}")
for lno, ctx in hits_0678:
    print(f"    line {lno}: {ctx[:100]}")
print(f"  Occurrences of '9/30' in source            : {len(hits_930)}")
for lno, ctx in hits_930[:8]:
    print(f"    line {lno}: {ctx[:100]}")
if len(hits_930) > 8:
    print(f"    ... and {len(hits_930) - 8} more")

print()
if not hits_0678:
    print("  [RESULT] '0.678' does NOT occur anywhere in the current source.")
    print("  Registry step (1) ['confirm both 9/30 and 0.678 actually co-occur'] cannot be")
    print("  confirmed against this file. All '9/30' occurrences found are part of the")
    print("  already-documented withdrawn-run narrative (\u00a710.7 / tab:feynman30-legacy),")
    print("  not paired with any '0.678' figure. Recommend closing FIX-N4 as not reproducible")
    print("  against current source, or asking the registry owner for the exact source location")
    print("  the original finding was based on.")
else:
    print("  [ACTION] '0.678' found -- proceed with registry steps (2) and (3): determine")
    print("  whether it's a mislabeled/stale figure or a distinct legitimate baseline, then")
    print("  add a detector regex once the correct value is known.")


In [ ]:
print(
    "NB-04 STATUS (2026-07-03 audit round)\n"
    "\n"
    "  FIX-N1  '71 cases' in Spearman footnote -- RESOLVED ('70 tasks' confirmed). \u2705\n"
    "  FIX-N2  Five-Layer vs Five-Stage terminology -- RESOLVED ('Five-Stage' throughout, confirmed). \u2705\n"
    "  FIX-N3  Nguyen-12 dual-threshold -- paper prose SETTLED (33.3%/91.7% stated directly, \u00a710.8);\n"
    "          seed=123 rerun file still not committed -- CI-gating stub stays open. \u23f3\n"
    "  FIX-C3  Feynman split-protocol saga -- RESOLVED on its 3rd re-open. Both prior 'fixes'\n"
    "          (106/180, then 71/90) were themselves stale duplicate-counting artifacts.\n"
    "          Live figures: 12/30 (40.0%) random 80/20, 13/30 (43.3%) PCA 40/60,\n"
    "          both self-verification-gated. TRUTH dict re-anchored; Step 6b checks result files. \u2705\n"
    "  FIX-D1  DeFi hybrid-attribution-bug -- OPEN, critical. Paper discloses corrected figures\n"
    "          inline (60.8% overall, -4.8pp hard-tier, 22/74 masked catastrophic) alongside the\n"
    "          pre-bug headline numbers. Step 6c adds footnote-aware checks; corrected DeFi result\n"
    "          file has not yet been located/committed for CI gating. \u26a0\ufe0f\n"
    "  FIX-N4  '9/30 vs 0.678' in \u00a710.7 -- OPEN, investigation stub. Step 6d searched the\n"
    "          current source and found NO occurrence of '0.678' anywhere; recommend the registry\n"
    "          entry be re-verified against its original source or closed as not reproducible. \u2753\n"
)
